In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn.linear_model as lm
import pandas as pd
from tqdm import trange

from sklearn.metrics import roc_auc_score

import mat73

In [ ]:
dir_name = '/home/austin/Aggression2/SingleRegionPredictive/'
fname = dir_name + 'SingleRegion_Aggression_data.mat'
my_dict = mat73.loadmat(fname)

In [ ]:
TrainsetMouse = my_dict['TrainsetMouse']
mbeh_all2 = my_dict['mbeh_all2']
mcond_all2 = my_dict['mcond_all2']
mouse_all2 = my_dict['mouse_all2']
mpow_all2 = my_dict['mpow_all2']
mpow_all3s = my_dict['mpow_all3s']
mtimecondbeh2 = my_dict['mtimecondbeh2']
mu_all3 = my_dict['mu_all3']
testsetMouse = my_dict['testsetMouse']


### Get index of training set

In [ ]:
N_samples = len(mouse_all2)

mice_all = []
for mouse in TrainsetMouse:
    mice_all.append(mouse[0])
for mouse in testsetMouse:
    mice_all.append(mouse[0])
mice_all = np.array(mice_all)

trainset = []
for i in range(len(TrainsetMouse)):
    trainset.append(TrainsetMouse[i][0])

train_idxs = np.zeros(N_samples)
mouse_idxs = np.zeros(N_samples)
for i in range(N_samples):
    if mouse_all2[i][0] == 'Mouse048':
        train_idxs[i] = -1
        continue
    if mouse_all2[i][0] in trainset:
        train_idxs[i] = 1
    mouse_idxs[i] = np.where(mice_all==mouse_all2[i][0])[0][0]
np.mean(train_idxs)

### Get positive and negative indexes of conditions

In [ ]:
idxs_pos = (mcond_all2==6)&(mbeh_all2==2)
idx_neg = ((mcond_all2==4)&(mbeh_all2>0))|((mcond_all2==8)&(mbeh_all2==2))
selection_indices = idxs_pos|idx_neg
print(np.mean(idxs_pos))
print(np.mean(idx_neg))
print(np.mean(selection_indices))

### Create task labels

In [ ]:
y = np.zeros(N_samples)
y[idxs_pos] = 1

In [ ]:
mpower_reduced = mpow_all2[:,:,selection_indices]
y_reduced = y[selection_indices]
train_idxs_reduced = train_idxs[selection_indices]

In [ ]:
y_train = y_reduced[train_idxs_reduced==1]
y_test = y_reduced[train_idxs_reduced==0]

In [ ]:
m_idx_unique_test = np.unique(mouse_idxs[train_idxs==0])
mouse_idxs_reduced = mouse_idxs[selection_indices]
m_test = mouse_idxs_reduced[train_idxs_reduced==0]

## Now fit all of the individual models

In [ ]:
region_list = ['IL','LHb','LSN','MDThal','MeA','NAc','OFC','PL','V1','VHipp','VMHvl']
names = ['Region'] + [testsetMouse[i][0] for i in range(9)]


In [ ]:
model_dict = {}

result_dict = {}

for i in range(11):
    for j in trange(i+1,11):
        XT1 = np.squeeze(mpower_reduced[:,i,:])
        XT2 = np.squeeze(mpower_reduced[:,j,:])
        X1 = np.transpose(XT1)
        X2 = np.transpose(XT2)
        X = np.hstack((X1,X2))
        X = X*10
        X[X>6] = 6
        
        key = region_list[i] + ' + ' + region_list[j]
        test_aucs = np.zeros(9)
    
        Xtrain = X[train_idxs_reduced==1]
        Xtest = X[train_idxs_reduced==0]
        model = lm.LogisticRegressionCV(max_iter=10000,n_jobs=6,random_state=42)
        model.fit(Xtrain,y_train)
        S_test = model.decision_function(Xtest)
        for k in range(9):
            test_aucs[k] = roc_auc_score(y_test[m_test==20+k],
                                S_test[m_test==20+k])
        
        model_dict[key] = model
        result_dict[key] = test_aucs
        
        
    

In [ ]:
keys = list(result_dict.keys())
vectors = np.array(list(result_dict.values()))
df = pd.DataFrame(vectors,index=keys).reset_index()
df.columns = names
df.to_csv('LR_FemaleVsAggression_2Region.csv',index=False)
import cloudpickle
with open('LR_FemaleVsAggression_twoRegion.p','wb') as f:
    cloudpickle.dump(model_dict,f)